# Dijkstra — prueba con un archivo

Implementación en C con el **algoritmo de Dial**: como los pesos son enteros de 1 a 100,
usa 101 cubetas circulares en vez de un heap, así que sacar el mínimo cuesta *O(1)*.
Soporta los límites del enunciado: `V ≤ 1.000.000`, `E ≤ 500.000`.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todas*. La última celda pide el
archivo de entrada; al subirlo se ejecuta y muestra la distancia y el tiempo.

Entrada: `V E` / `s t` / V líneas de etiquetas / E líneas `u v w`.
Salida: la distancia (o `No hay camino`), y el tiempo que tardó.

### 1. El código

In [ ]:
%%writefile dijkstra.c
/* Dijkstra con cubetas de Dial (pesos enteros 1..100), todo en un archivo.
 * Compilar:  gcc -O2 -o dijkstra dijkstra.c
 * Ejecutar:  ./dijkstra entrada.txt      (imprime la distancia y el tiempo)
 * Formato:   "V E" / "s t" / V lineas de etiquetas / E lineas "u v w".
 */
#include <time.h>
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <fcntl.h>
#include <sys/stat.h>
#include <sys/mman.h>

#define UNDIRECTED 1              /* "arista entre u y v" => doble sentido */
#define WMAX 100
#define INF 0xffffffffu

struct Node { unsigned dist; int head; };   /* dist y head juntos: una linea de cache */
struct Arc  { int tw; int nx; };            /* tw = (destino << 7) | peso */

struct Graph {
    unsigned V, E, s, t;
    size_t narcs;
    struct Node *node;   /* indices 1..V */
    struct Arc *arc;
};

static void die(const char *m) { fprintf(stderr, "error: %s\n", m); exit(1); }

static inline unsigned rd(const char **pp, const char *end) {
    const char *p = *pp;
    while (p < end && (unsigned)(*p - '0') > 9u) p++;
    unsigned x = 0, c;
    while (p < end && (c = (unsigned)(*p - '0')) <= 9u) { x = x * 10 + c; p++; }
    *pp = p;
    return x;
}

/* Salta n lineas contando '\n' en bloques de 64 (el compilador lo vectoriza):
   un memchr por linea costaba 1M de llamadas = ~5 ms con V=1.000.000. */
static const char *skip_lines(const char *p, const char *end, long n) {
    while (n > 0 && end - p >= 64) {
        long c = 0;
        for (int i = 0; i < 64; i++) c += (p[i] == '\n');
        if (c >= n) break;   /* el salto n-esimo esta en este bloque: termina byte a byte */
        n -= c; p += 64;
    }
    while (n > 0 && p < end) { if (*p == '\n') n--; p++; }
    return p;
}

/* Paginas grandes (2 MB): menos fallos de pagina al inicializar los 8 MB de
   nodos y menos fallos de TLB en los accesos aleatorios durante el parseo. */
static void *gr_alloc(size_t n) {
    void *m = mmap(NULL, n, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (m == MAP_FAILED) die("sin memoria");
    madvise(m, n, MADV_HUGEPAGE);
    return m;
}

/* libera lo que reservo gr_parse (solo lo usa bench.c al repetir el parseo) */
static void gr_free(struct Graph *g) {
    if (g->node) munmap(g->node, (size_t)(g->V + 1) * sizeof *g->node);
    if (g->arc) munmap(g->arc, (g->narcs ? g->narcs : 1) * sizeof *g->arc);
    g->node = NULL; g->arc = NULL;
}

static char *slurp_fd(int fd, size_t *len) {
    size_t cap = 1u << 22, n = 0;
    char *b = malloc(cap);
    if (!b) return NULL;
    for (;;) {
        if (n == cap) { cap *= 2; char *tb = realloc(b, cap); if (!tb) { free(b); return NULL; } b = tb; }
        ssize_t r = read(fd, b + n, cap - n);
        if (r <= 0) break;
        n += (size_t)r;
    }
    *len = n;
    return b;
}

/* mmap del archivo (o lectura completa si es stdin/pipe) */
static void gr_load(const char *path, const char **pp, const char **pend) {
    if (!path) {
        size_t n; char *b = slurp_fd(0, &n);
        if (!b) die("sin memoria");
        *pp = b; *pend = b + n;
        return;
    }
    int fd = open(path, O_RDONLY);
    if (fd < 0) die("no se pudo abrir el archivo");
    struct stat st;
    char *m = MAP_FAILED;
    size_t sz = 0;
    if (!fstat(fd, &st) && S_ISREG(st.st_mode)) {
        sz = (size_t)st.st_size;
        m = sz ? mmap(NULL, sz, PROT_READ, MAP_PRIVATE | MAP_POPULATE, fd, 0) : (char *)"";
    }
    if (m == MAP_FAILED) {
        size_t n; char *b = slurp_fd(fd, &n);
        if (!b) die("sin memoria");
        *pp = b; *pend = b + n;
    } else {
        *pp = m; *pend = m + sz;
    }
    close(fd);
}

/* parsea cabecera + etiquetas + aristas y construye las listas de adyacencia */
static void gr_parse(const char *p, const char *end, struct Graph *g) {
    g->V = rd(&p, end); g->E = rd(&p, end);
    g->s = rd(&p, end); g->t = rd(&p, end);
    unsigned V = g->V, E = g->E;
    if (V < 1 || g->s < 1 || g->s > V || g->t < 1 || g->t > V) die("V, s o t fuera de rango");
    if (g->s == g->t) { g->node = NULL; g->arc = NULL; g->narcs = 0; return; }   /* nada que parsear */

    p = skip_lines(p, end, (long)V + 1);   /* resto de la linea 2 + V etiquetas */

    g->narcs = (size_t)E * (UNDIRECTED ? 2 : 1);
    g->node = gr_alloc((size_t)(V + 1) * sizeof *g->node);
    g->arc = gr_alloc((g->narcs ? g->narcs : 1) * sizeof *g->arc);
    if (!g->node || !g->arc) die("sin memoria");
    memset(g->node, 0xff, (size_t)(V + 1) * sizeof *g->node);   /* dist=INF, head=-1 */

    struct Node *node = g->node;
    struct Arc *arc = g->arc;
    int na = 0;
    for (unsigned i = 0; i < E; i++) {
        unsigned u = rd(&p, end), v = rd(&p, end), w = rd(&p, end);
        if (u < 1 || u > V || v < 1 || v > V || w < 1 || w > WMAX) die("arista fuera de rango");
        arc[na].tw = (int)((v << 7) | w); arc[na].nx = node[u].head; node[u].head = na++;
#if UNDIRECTED
        arc[na].tw = (int)((u << 7) | w); arc[na].nx = node[v].head; node[v].head = na++;
#endif
    }
}

#define NB (WMAX + 1)   /* 101 cubetas circulares de Dial */

struct Ent { int node; int next; };   /* entrada de cubeta (lista enlazada en pool) */

static inline int ctz128(unsigned __int128 x) {
    unsigned long long lo = (unsigned long long)x;
    return lo ? __builtin_ctzll(lo) : 64 + __builtin_ctzll((unsigned long long)(x >> 64));
}

/* ---- 1) Dial con mascara de bits (salta cubetas vacias en O(1)) ---- */
static unsigned solve_dial(struct Node *node, const struct Arc *arc,
                           unsigned s, unsigned t, struct Ent *pool) {
    int bhead[NB];
    memset(bhead, 0xff, sizeof bhead);
    unsigned __int128 mask = 1;
    int np = 0, b = 0;
    unsigned curd = 0;

    node[s].dist = 0;
    pool[np].node = (int)s; pool[np].next = -1; bhead[0] = np++;

    while (mask) {
        unsigned __int128 hi = mask >> b;                  /* cubetas en [b, 100] */
        int k = hi ? ctz128(hi) : NB - b + ctz128(mask);   /* si no, da la vuelta */
        curd += (unsigned)k;
        b += k; if (b >= NB) b -= NB;

        for (int e = bhead[b]; e >= 0; e = bhead[b]) {
            bhead[b] = pool[e].next;
            unsigned u = (unsigned)pool[e].node;
            if (node[u].dist != curd) continue;            /* entrada obsoleta */
            if (u == t) return curd;
            for (int a = node[u].head; a >= 0; a = arc[a].nx) {
                unsigned tw = (unsigned)arc[a].tw, v = tw >> 7, w = tw & 127;
                unsigned nd = curd + w;
                if (nd < node[v].dist) {
                    node[v].dist = nd;
                    int b2 = b + (int)w; if (b2 >= NB) b2 -= NB;   /* w<=100 => b2!=b */
                    pool[np].node = (int)v; pool[np].next = bhead[b2]; bhead[b2] = np++;
                    mask |= (unsigned __int128)1 << b2;
                }
            }
        }
        mask &= ~((unsigned __int128)1 << b);
    }
    return INF;
}

/* ------------------------------ programa ------------------------------ */

#define NO_PATH "No hay camino"   /* cambiar por "-1" si lo piden asi */

static double ms(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1e3 + t.tv_nsec / 1e6;
}

int main(int argc, char **argv) {
    double t0 = ms();
    const char *p, *end;
    gr_load(argc > 1 ? argv[1] : NULL, &p, &end);   /* sin argumento: lee stdin */

    struct Graph g;
    gr_parse(p, end, &g);
    double t1 = ms();

    unsigned d = 0;   /* s == t => 0, y gr_parse no llego a construir el grafo */
    if (g.s != g.t) {
        struct Ent *pool = malloc((g.narcs + 2) * sizeof *pool);
        if (!pool) die("sin memoria");
        d = solve_dial(g.node, g.arc, g.s, g.t, pool);
    }
    double t2 = ms();

    if (d == INF) puts(NO_PATH);
    else printf("%u\n", d);

    /* el tiempo va a stderr: por stdout sale solo la distancia */
    fprintf(stderr, "tiempo: parseo %.3f ms | dijkstra %.3f ms | total %.3f ms\n",
            t1 - t0, t2 - t1, t2 - t0);
    return 0;
}

### 2. Compilar y comprobar con el ejemplo del enunciado (debe dar **21**)

In [ ]:
!gcc -O2 -o dijkstra dijkstra.c && echo "compilado OK"
!printf '10 2\n3 7\n1\n2\n3\n4\n5\n6\n7\n8\n9\n10\n3 5 14\n5 7 7\n' > ejemplo.txt
!./dijkstra ejemplo.txt

### 3. Probar con el archivo del profesor

In [ ]:
from google.colab import files
subido = files.upload()
nombre = next(iter(subido))
!./dijkstra "$nombre"